In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import torch.optim as optim

from smallcnn import SmallImageCNN
from pathlib import Path

ImportError: cannot import name 'SmallCNN' from 'smallcnn' (/home/gpu_03/class-agnostic/smallcnn.py)

In [ ]:


# Define the "recipe" for your images
data_transforms = transforms.Compose([
    transforms.Resize((64, 64)),         # Ensure all inputs are exactly 64x64
    transforms.RandomHorizontalFlip(),   # Simple augmentation to prevent overfitting
    transforms.ToTensor(),               # Convert to pixel values 0-1
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]) # Standard normalization
])

# Load images from folders
train_dataset = datasets.ImageFolder(root='VisDrone-objects/train', transform=data_transforms)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

val_dataset = datasets.ImageFolder(root='VisDrone-objects/val', transform=data_transforms)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:

# 1. Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SmallImageCNN(num_classes=10).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()


NameError: name 'SmallImageCNN' is not defined

In [ ]:
def validate_model(model, val_loader, criterion, device):
    model.eval()  # Set to evaluation mode
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():  # Turn off gradients to save memory and speed up
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            
            # Calculate accuracy
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    avg_loss = val_loss / len(val_loader.dataset)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

# Usage inside your training loop:
# val_loss, val_acc = validate_model(model, val_loader, criterion, device)
# print(f"Validation Accuracy: {val_acc:.2f}%")

In [ ]:

log_dir = Path('smallcnn_logs')
log_dir.mkdir(exist_ok=True)

In [ ]:


# Configuration
num_epochs = 1
best_val_acc = 0.0

for epoch in range(num_epochs):
    # --- TRAINING PHASE ---
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()

    # --- VALIDATION PHASE ---
    # Using the validate_model function from the previous step
    val_loss, val_acc = validate_model(model, val_loader, criterion, device)
    
    print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {running_loss/len(train_loader):.4f} | Val Acc: {val_acc:.2f}%")
    
    # --- SAVING PROGRESS ---
    # 1. Save the "Latest" version (overwrites every epoch)
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss.item(),
    }, f'{log_dir}/last_checkpoint.pth')
    
    # 2. Save the "Best" version (only if accuracy improved)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), f'{log_dir}/best_model.pth')
        print(f"--> New best model saved with {val_acc:.2f}% accuracy!")
